In [ ]:
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [ ]:
data = pd.read_csv('/kaggle/input/notebooks/amritanshukush/lendingclub-cleaning/lending_club_cleansed.csv')
data.head()

In [ ]:
print(f"Dataset shape before splitting: {data.shape}")

# ---------------------------------------------------------
# STEP 1: Define Features (X) and Target (y)
# ---------------------------------------------------------
# We drop 'default' from X because it is the answer key.
X = data.drop(columns=['default'])
y = data['default']

# ---------------------------------------------------------
# STEP 2: Train/Test Split
# ---------------------------------------------------------
# We split the data: 80% for training the model, 20% for testing its accuracy.
# stratify=y ensures that the 80/20 split maintains the exact same ratio of defaults.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# ---------------------------------------------------------
# STEP 3: Handle Class Imbalance
# ---------------------------------------------------------
# There are far more "Good" loans (0) than "Default" loans (1).
# We calculate the exact ratio so XGBoost pays more attention to the defaults.
num_negative = (y_train == 0).sum()
num_positive = (y_train == 1).sum()
imbalance_ratio = num_negative / num_positive

print(f"Class Imbalance Ratio (Good / Default): {imbalance_ratio:.2f}")

In [ ]:
# ---------------------------------------------------------
# STEP 4: Initialize and Train XGBoost Classifier
# ---------------------------------------------------------
print("Training XGBoost Model... (This may take a minute or two)")

xgb_model = xgb.XGBClassifier(
    n_estimators=1000,            # Number of trees
    max_depth=7,                 # How deep each tree can go (prevents overfitting)
    learning_rate=0.03,           # How aggressively it learns
    scale_pos_weight=imbalance_ratio, # Crucial: Punishes the model harder for missing defaults
    eval_metric='auc',           # Optimize for Area Under the ROC Curve
    random_state=42,
    n_jobs=-1                    # Use all available CPU cores
)

# Fit the model to the training data
xgb_model.fit(X_train, y_train)
print("Training Complete!")

# ---------------------------------------------------------
# STEP 5: Evaluate Model Performance
# ---------------------------------------------------------
# We generate raw probabilities (e.g., 0.15 chance of default)
# and hard predictions (0 or 1 based on a 0.5 threshold)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
y_pred = xgb_model.predict(X_test)

# ROC-AUC is the industry standard for credit risk. 
# 0.5 is a random coin flip. 0.7+ is good. 0.8+ is excellent.
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"\n--- Model Evaluation ---")
print(f"ROC-AUC Score: {auc_score:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
# Top Left: True Negatives (Correctly predicted Paid)
# Top Right: False Positives (Predicted Default, but actually Paid)
# Bottom Left: False Negatives (Predicted Paid, but actually Default - DANGEROUS!)
# Bottom Right: True Positives (Correctly predicted Default)
print(confusion_matrix(y_test, y_pred))

In [ ]:
# ---------------------------------------------------------
# STEP 6: Save the Model and Feature List
# ---------------------------------------------------------
# We save the trained model so we can load it later in our Dash App.
# We also save the exact column names so our app knows what inputs the model expects.

joblib.dump(xgb_model, 'pd_model.pkl')
joblib.dump(X_train.columns.tolist(), 'model_features.pkl')

print("\nModel saved as 'pd_model.pkl'")
print("Feature list saved as 'model_features.pkl'")

In [ ]:
sample_portfolio = data.sample(n=20, random_state=42)
sample_portfolio.to_csv('live_portfolio.csv', index=False)